# Q-Folio: constraint-preserving QAOA for portfolio selection

**Qiskit Fall Fest 2026 · Industry Track I2 (Portfolio Optimizer)** · Achintya Akella

The question: *pick exactly k of n stocks for an equal-weight basket that balances return and risk*.
We solve it four classical ways and two quantum ways (penalty-QAOA and XY-QAOA), compare them fairly,
and then run the quantum circuit on a noisy simulator and on a real IBM Heron device.

**Structure:** Predict → Build → Benchmark → Explain.
All logic lives in the `qportfolio` package; this notebook only calls it. Results come from
`results/*.json` (written by `scripts/run_benchmark.py` and `scripts/run_noisy.py`); set
`QFOLIO_RECOMPUTE=1` to recompute the headline instance here instead.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

if Path.cwd().name == "notebooks":  # run from the repo root so relative paths work
    os.chdir("..")

import numpy as np
import pandas as pd

from qportfolio import viz
from qportfolio.benchmark import METHODS, run_instance, scaling_rows, summarise
from qportfolio.circuits import build_qaoa, dicke_state
from qportfolio.classical import brute_force, greedy, simulated_annealing
from qportfolio.config import is_fast_mode, load_config
from qportfolio.demo import basket, block_diagram, default_problem
from qportfolio.hardware import cost_gap
from qportfolio.io import load_result
from qportfolio.metrics import evaluate_counts
from qportfolio.problem import bitstring_to_x, x_to_bitstring
from qportfolio.qaoa import optimize, sample

RECOMPUTE = os.environ.get("QFOLIO_RECOMPUTE") == "1"
cfg = load_config()
print(f"seed {cfg.seed} · fast mode {is_fast_mode()} · recompute {RECOMPUTE}")

## 1 · Business problem

Indian wealth-tech apps sell ready-made "model baskets" of a few stocks to retail investors.
Choosing *which* k stocks to hold is a combinatorial problem: with n candidates there are C(n, k)
baskets, and the trade-off between return and risk has to be scored for every one of them.

**Prediction before we build:** a quantum optimiser that ignores the "exactly k" rule will waste
most of its shots on invalid baskets. Building the rule into the circuit should fix that.

## 2 · Data

Daily adjusted closing prices of six large NSE stocks (2023–2025), cached in `data/prices.csv`
(no network needed). From daily log returns we get the annualised mean return μ and covariance Σ.

In [ ]:
p = default_problem()
stats = pd.DataFrame(
    {"annual return μ": p.mu, "annual volatility": np.sqrt(np.diag(p.sigma))}, index=p.labels
)
print(f"n = {p.n} assets, pick k = {p.k}, risk aversion q = {p.q}")
stats.style.format("{:.1%}")

## 3 · Formulation

Choose x ∈ {0,1}ⁿ with exactly k ones and minimise

$$C(x) = q\,x^\top\Sigma x - \mu^\top x \quad\text{s.t.}\quad \textstyle\sum_i x_i = k.$$

* **QUBO (penalty form):** add λ(Σx − k)², which gives a matrix Q with C_λ(x) = xᵀQx + λk².
  λ = the range of feasible costs, the same for every penalty-based method.
* **Ising form:** substitute x_i = (1 − z_i)/2 to get H = Σ J_ij Z_i Z_j + Σ h_i Z_i (normalised by max|Q|).

The check below confirms the QUBO matches C(x) + penalty on all 2ⁿ bitstrings (acceptance test AC1).

In [ ]:
lam = p.default_penalty()
Q, const = p.to_qubo(lam)
worst = 0.0
for i in range(2**p.n):
    x = bitstring_to_x(format(i, f"0{p.n}b"))
    worst = max(worst, abs(x @ Q @ x + const - (p.cost(x) + lam * (x.sum() - p.k) ** 2)))
H = p.to_ising(0.0)
print(f"λ = {lam:.4f}; max |QUBO − (C + penalty)| over {2**p.n} strings = {worst:.1e}")
print(f"Ising Hamiltonian: {len(H)} Pauli terms (Z and ZZ)")

## 4 · Classical baselines

* **Brute force** scores all C(6,3) = 20 baskets: exact, and instant at this size.
* **Greedy** adds the best next stock until k are chosen ("the analyst with a spreadsheet").
* **Simulated annealing** samples the *same* penalty QUBO, so its P(optimal) over reads is the fair
  comparison for QAOA's P(optimal) over shots.

In [ ]:
best, costs = brute_force(p)
g = greedy(p)
c = cfg.raw["classical"]
sa = simulated_annealing(p, lam, c["sa_sweeps"], cfg.seed, reads=c["sa_reads"])
sa_counts = {}
for x in sa.samples:
    b = x_to_bitstring(x)
    sa_counts[b] = sa_counts.get(b, 0) + 1
sa_m = evaluate_counts(sa_counts, p, costs, best.x)

ranked = sorted(zip(p.feasible_set(), costs, strict=True), key=lambda t: t[1])
top = pd.DataFrame(
    [{"basket": basket(p, x_to_bitstring(x)), "cost C(x)": cst} for x, cst in ranked[:5]],
    index=range(1, 6),
)
gap = cost_gap(p)
print(f"brute force optimum: {basket(p, x_to_bitstring(best.x))}  ({best.seconds * 1e3:.2f} ms)")
print(f"greedy:              {basket(p, x_to_bitstring(g.x))}")
print(f"simulated annealing: P(optimal) over {c['sa_reads']} reads = {sa_m['p_optimal']:.2f}")
print(
    f"optimum vs runner-up cost gap = {gap['gap']:.4f} "
    f"({gap['gap'] / gap['cost_range']:.1%} of the cost range): a near-tie"
)
top

## 5 · Penalty-QAOA, and why it struggles

Standard QAOA starts in |+⟩ⁿ (all 64 bitstrings, including the 44 invalid ones) and relies on the
penalty term to push probability towards valid baskets. A large λ makes the valid-basket energy
differences tiny compared with the penalty, so the optimiser mainly learns "be valid" and barely
learns "be good". Results below are from `results/benchmark.json`.

In [ ]:
bench_path = Path("results/benchmark.json")
if RECOMPUTE or not bench_path.exists():
    q = cfg.raw["qaoa"]
    settings = {
        "restarts": q["restarts"],
        "maxiter": q["maxiter"],
        "shots": q["shots"],
        "sa_sweeps": c["sa_sweeps"],
        "sa_reads": c["sa_reads"],
    }
    head = run_instance(p, settings, cfg.seed, keep_counts=True)
    head.update(id=0, kind="headline", tickers=list(cfg.raw["data"]["tickers"]))
    bench = {"instances": [head], "summary": summarise([head])}
    print("recomputed the headline instance (robustness set = this instance only)")
else:
    bench = load_result(bench_path)
    n_inst, created = len(bench["instances"]), bench["meta"]["created"]
    print(f"loaded {bench_path} ({n_inst} instances, created {created})")
head = bench["instances"][0]
cols = ["p_feasible", "p_optimal", "p_top2", "approx_ratio"]
pd.DataFrame({m: head["methods"][m] for m in METHODS if "qaoa" in m}).T[cols].astype(float).round(3)

In [ ]:
_ = viz.feasibility(bench)

## 6 · XY-QAOA: build the constraint into the circuit

Two changes make every shot a valid basket:

1. **Dicke state** |D(n,k)⟩: start in an equal superposition of *only* the C(n,k) weight-k strings
   (Bärtschi & Eidenbenz 2019).
2. **XY-ring mixer**: `XXPlusYYGate` on neighbouring qubits only swaps |01⟩ ↔ |10⟩, so it moves
   probability between baskets without ever changing how many stocks are held.

The cost layer exp(−iγH) is diagonal, so it cannot change the weight either. No penalty is needed (λ = 0).

In [ ]:
print(block_diagram(p).draw(output="text", fold=120))

In [ ]:
print("Dicke(4, 2) state preparation, decomposed:")
print(dicke_state(4, 2).draw(output="text", fold=120))

A live p = 1 run on the ideal simulator (angles optimised on the exact ⟨H⟩, then sampled):

In [ ]:
q = cfg.raw["qaoa"]
ansatz, H_xy = build_qaoa(p, reps=1, variant="xy")
res = optimize(ansatz, H_xy, restarts=q["restarts"], maxiter=q["maxiter"], seed=cfg.seed)
counts = sample(ansatz, res.params, shots=q["shots"], seed=cfg.seed)
m = evaluate_counts(counts, p, costs, best.x)
print(
    f"XY-QAOA p=1: P(feasible) = {m['p_feasible']:.3f}, P(optimal) = {m['p_optimal']:.3f}, "
    f"P(top-2) = {m['p_top2']:.3f}, approx. ratio = {m['approx_ratio']:.3f} "
    f"({res.nfev} energy evaluations, {res.seconds:.1f} s)"
)
print(f"random guess:  P(optimal) = {1 / len(costs):.3f}, P(top-2) = {2 / len(costs):.3f}")

## 7 · Benchmark results

Fair-comparison rules (docs/05): same objective, same λ, same scoring function for every method;
wall time includes QAOA angle optimisation. Bars are the headline NSE instance; dots are mean ± std
over the robustness set (random 6-stock subsets of a 10-stock universe × q ∈ {0.25, 0.5, 1.0}).

In [ ]:
_ = viz.headline(bench)

In [ ]:
rows = {}
for name in METHODS:
    s = bench["summary"][name]
    rows[name] = {
        "P(optimal)": f"{s['p_optimal_mean']:.3f} ± {s['p_optimal_std']:.3f}",
        "P(top-2)": f"{s['p_top2_mean']:.3f} ± {s['p_top2_std']:.3f}",
        "approx. ratio": f"{s['approx_ratio_mean']:.3f} ± {s['approx_ratio_std']:.3f}",
        "P(feasible)": f"{s['p_feasible_mean']:.3f}",
        "top-1 optimal": "–"
        if s["top1_is_optimal_share"] is None
        else f"{s['top1_is_optimal_share']:.0%}",
        "wall time (s)": f"{s['seconds_mean']:.3g}",
    }
print(f"robustness set: {bench['summary']['brute_force']['instances']} instances, mean ± std")
pd.DataFrame(rows).T

In [ ]:
_ = viz.distribution(bench)

In [ ]:
_ = viz.frontier(bench)

**What we say out loud:** brute force is exact and fastest at this size; simulated annealing finds
the optimum reliably; XY-QAOA is far better than penalty-QAOA and random and keeps 100 % feasibility.
Statevector simulation time is not quantum runtime. No quantum advantage is claimed.

## 8 · Noise-aware engineering and real hardware

Hardware instance: 4 stocks, pick 2 (shallow enough for today's devices). We transpile for IBM
Heron devices, run on noisy simulators (FakeTorino, FakeFez) and on the real **ibm_kingston**, then
apply two corrections:

* **Symmetry post-selection:** the XY circuit can only output weight-k strings, so any other string
  *must* be an error. Discarding them is free error detection (valid only for the XY circuit).
* **Readout mitigation:** per-qubit 2×2 confusion matrices from two calibration circuits, inverted.

In [ ]:
noisy = load_result("results/noisy.json")
hw = load_result("results/hardware/db3b1bimb58s7387e0jg.json")
dry = load_result("results/hardware/dry_run_fake_torino.json")
_ = viz.transpile(noisy)

In [ ]:
table = {"ideal (statevector)": {f"p={r}": m["p_optimal"] for r, m in noisy["ideal"].items()}}
for name, data in noisy["backends"].items():
    for stage in ("raw", "postselected", "post_mitigated"):
        table[f"{name} {stage}"] = {f"p={r['reps']}": r[stage]["p_optimal"] for r in data["runs"]}
real = {r["reps"]: r for r in hw["runs"]}
table[f"{hw['backend']} raw"] = {f"p={k}": r["raw"]["p_optimal"] for k, r in real.items()}
table[f"{hw['backend']} postselected"] = {
    f"p={k}": r["postselected"]["p_optimal"] for k, r in real.items()
}
table[f"{hw['backend']} post_mitigated"] = {
    f"p={k}": r["mitigated"]["p_optimal_postselected"] for k, r in real.items()
}
print(f"P(optimal) down the noise ladder; random guess = {noisy['p_random']:.3f}")
pd.DataFrame(table).T.round(3)

In [ ]:
# The ideal rung uses the same statevector-trained angles that were sent to ibm_kingston.
_ = viz.noise_ladder(noisy["ideal"], dry, hw)
print(
    f"IBM job {hw['job_id']} on {hw['backend']}: {hw['shots']} shots, "
    f"{hw['usage']['qpu_charge_time_seconds']} s of QPU time"
)

## 9 · Limits and next steps

* **Scale:** C(6,3) = 20 baskets. Brute force wins easily here and will until n ≈ 40; good classical
  heuristics handle hundreds of assets. This is a methods study, not a speed race.
* **What would need to be true for QAOA to matter:** much lower-error (or fault-tolerant) hardware for
  the depth larger n needs, evidence that P(optimal) decays more slowly than classical heuristics on
  hard instances, and cheap parameter setting at scale (parameter transfer).
* **Model:** equal weights and a static covariance; no transaction costs or sector limits yet.
  Sector limits are more constraints that XY-style mixers can also respect.
* **Takeaway:** building the constraint into the circuit is necessary even at toy scale, and the same
  symmetry gives free error detection on real hardware.

In [ ]:
_ = viz.scaling(scaling_rows(), hw)